In [ ]:
import sys, platform, time
import numpy as np
import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

In [ ]:
import nltk
print("NLTK data paths:")
for p in nltk.data.path:
    print(" -", p)

In [ ]:
import os, nltk

target_dir = nltk.data.path[0]
os.makedirs(target_dir, exist_ok=True)
print("Downloading into:", target_dir)

nltk.download("punkt", download_dir=target_dir)
nltk.download("punkt_tab", download_dir=target_dir)
nltk.download("reuters", download_dir=target_dir)

In [ ]:
import nltk
from nltk.data import find

print("punkt:", find("tokenizers/punkt"))
print("punkt_tab:", find("tokenizers/punkt_tab/english/"))

In [ ]:
from nltk.corpus import reuters
from nltk.tokenize import sent_tokenize, word_tokenize

fileids = reuters.fileids()
print("Total Reuters docs:", len(fileids))

SANITY_DOCS = 2000
subset_ids = fileids[:SANITY_DOCS]

sentences = []
for fid in subset_ids:
    raw = reuters.raw(fid)
    for sent in sent_tokenize(raw):
        tokens = [w.lower() for w in word_tokenize(sent)]
        tokens = [w for w in tokens if w.isalpha()]
        if len(tokens) >= 3:
            sentences.append(tokens)

print("Sanity sentences:", len(sentences))
print("Example sentence:", sentences[0][:20])

In [ ]:
from collections import Counter

# Hyperparams
MIN_COUNT = 5
MAX_VOCAB = 20000

# Count words
word_counts = Counter(w for sent in sentences for w in sent)

# Keep words above MIN_COUNT
filtered = [(w, c) for w, c in word_counts.items() if c >= MIN_COUNT]
filtered.sort(key=lambda x: x[1], reverse=True)

# Cap vocab
filtered = filtered[:MAX_VOCAB]

# Build mappings (reserve 0 for UNK)
word2id = {"<UNK>": 0}
id2word = {0: "<UNK>"}

for i, (w, c) in enumerate(filtered, start=1):
    word2id[w] = i
    id2word[i] = w

vocab_size = len(word2id)
print("Raw unique words:", len(word_counts))
print("Vocab size (with <UNK>):", vocab_size)

# Encode sentences as IDs
corpus_ids = []
unk_count = 0
total_tokens = 0

for sent in sentences:
    ids = []
    for w in sent:
        total_tokens += 1
        wid = word2id.get(w, 0)
        if wid == 0:
            unk_count += 1
        ids.append(wid)
    corpus_ids.append(ids)

print("Total tokens:", total_tokens)
print("UNK tokens:", unk_count, f"({unk_count/total_tokens:.2%})")
print("Example encoded sentence:", corpus_ids[0][:20])

In [ ]:
import numpy as np

WINDOW_SIZE = 2  # DYNAMIC - can be changed

skip_grams = []
for sent in corpus_ids:
    if len(sent) < 2 * WINDOW_SIZE + 1:
        continue
    for center_i in range(WINDOW_SIZE, len(sent) - WINDOW_SIZE):
        center = sent[center_i]
        for j in range(center_i - WINDOW_SIZE, center_i + WINDOW_SIZE + 1):
            if j == center_i:
                continue
            context = sent[j]
            skip_grams.append((center, context))

skip_grams = np.array(skip_grams, dtype=np.int64)
print("skip_grams shape:", skip_grams.shape)
print("Example pairs (center, context):", skip_grams[:5])

In [ ]:
import torch

BATCH_SIZE = 256

def get_batch(batch_size=BATCH_SIZE, device=device):
    idx = np.random.randint(0, len(skip_grams), size=batch_size)
    batch = skip_grams[idx]
    centers = torch.tensor(batch[:, 0], dtype=torch.long, device=device)
    contexts = torch.tensor(batch[:, 1], dtype=torch.long, device=device)
    return centers, contexts

# Quick sanity check
c, x = get_batch()
print("centers:", c.shape, c.dtype, c.device)
print("contexts:", x.shape, x.dtype, x.device)
print("sample:", c[:5].tolist(), x[:5].tolist())

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
np.random.seed(42)

class SkipGramSoftmax(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, emb_dim)
        self.out_linear = nn.Linear(emb_dim, vocab_size, bias=False)

    def forward(self, center_ids):
        v = self.in_embed(center_ids)         # [B, D]
        logits = self.out_linear(v)           # [B, V]
        return logits

EMB_DIM = 50
model_sg = SkipGramSoftmax(vocab_size=vocab_size, emb_dim=EMB_DIM).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_sg.parameters(), lr=0.01)

# Tiny sanity training: 200 steps
model_sg.train()
losses = []
for step in range(200):
    centers, contexts = get_batch()
    logits = model_sg(centers)
    loss = criterion(logits, contexts)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"step {step+1}/200 loss={loss.item():.4f}")

print("loss first/last:", losses[0], losses[-1])

In [ ]:
import time

FINAL_STEPS = 3000

model_sg.train()
t0 = time.time()

loss_log = []
for step in range(FINAL_STEPS):
    centers, contexts = get_batch()
    logits = model_sg(centers)
    loss = criterion(logits, contexts)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_log.append(loss.item())
    if (step + 1) % 500 == 0:
        avg = float(np.mean(loss_log[-500:]))
        print(f"step {step+1}/{FINAL_STEPS} avg_loss(last500)={avg:.4f}")

t1 = time.time()
train_time = t1 - t0
final_avg_loss = float(np.mean(loss_log[-500:]))

print("\nSkip-gram Softmax DONE")
print("time_sec:", train_time)
print("final_avg_loss:", final_avg_loss)

# Save embeddings
sg_embeddings = model_sg.in_embed.weight.detach().to("cpu").numpy()
np.save("embeddings/sg_softmax_embeddings.npy", sg_embeddings)
print("saved: sg_softmax_embeddings.npy", sg_embeddings.shape)

In [ ]:
# Build unigram distribution with 0.75 power
power = 0.75

id_counts = np.zeros(vocab_size, dtype=np.int64)
id_counts[0] = 0  # keep <UNK> out of negative sampling

for w, wid in word2id.items():
    if wid == 0:
        continue
    id_counts[wid] = word_counts[w]

# Probabilities
p = id_counts.astype(np.float64) ** power
p_sum = p.sum()
p = p / p_sum

# Unigram table
TABLE_SIZE = 200000
unigram_table = np.random.choice(np.arange(vocab_size), size=TABLE_SIZE, p=p)

print("unigram_table size:", unigram_table.shape)
print("table id min/max:", unigram_table.min(), unigram_table.max())
print("contains UNK (0)?", (unigram_table == 0).any())

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

NEG_K = 5
EMB_DIM = 50
BATCH_SIZE = 256

class SkipGramNEG(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, emb_dim)
        self.out_embed = nn.Embedding(vocab_size, emb_dim)

    def forward(self, center_ids, pos_context_ids, neg_context_ids):
        v = self.in_embed(center_ids)                 # [B, D]
        u_pos = self.out_embed(pos_context_ids)       # [B, D]
        u_neg = self.out_embed(neg_context_ids)       # [B, K, D]

        # positive loss
        pos_score = torch.sum(u_pos * v, dim=1)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-9)

        # negative loss
        neg_score = torch.bmm(u_neg, v.unsqueeze(2)).squeeze(2)
        neg_loss = -torch.sum(torch.log(torch.sigmoid(-neg_score) + 1e-9), dim=1)

        return (pos_loss + neg_loss).mean()

model_neg = SkipGramNEG(vocab_size=vocab_size, emb_dim=EMB_DIM).to(device)
optimizer_neg = optim.Adam(model_neg.parameters(), lr=0.01)

def get_batch_neg(batch_size=BATCH_SIZE, k=NEG_K, device=device):
    idx = np.random.randint(0, len(skip_grams), size=batch_size)
    batch = skip_grams[idx]
    centers = torch.tensor(batch[:, 0], dtype=torch.long, device=device)
    pos = torch.tensor(batch[:, 1], dtype=torch.long, device=device)

    # sample negatives from unigram table
    neg_np = unigram_table[np.random.randint(0, len(unigram_table), size=(batch_size, k))]
    pos_np = batch[:, 1]
    mask = (neg_np == pos_np[:, None])
    while mask.any():
        neg_np[mask] = unigram_table[np.random.randint(0, len(unigram_table), size=mask.sum())]
        mask = (neg_np == pos_np[:, None])

    neg = torch.tensor(neg_np, dtype=torch.long, device=device)
    return centers, pos, neg

# Sanity training
model_neg.train()
losses = []
for step in range(200):
    centers, pos, neg = get_batch_neg()
    loss = model_neg(centers, pos, neg)

    optimizer_neg.zero_grad()
    loss.backward()
    optimizer_neg.step()

    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"step {step+1}/200 loss={loss.item():.4f}")

print("loss first/last:", losses[0], losses[-1])

In [ ]:
FINAL_STEPS_NEG = 6000

model_neg.train()
t0 = time.time()

loss_log = []
for step in range(FINAL_STEPS_NEG):
    centers, pos, neg = get_batch_neg()
    loss = model_neg(centers, pos, neg)

    optimizer_neg.zero_grad()
    loss.backward()
    optimizer_neg.step()

    loss_log.append(loss.item())
    if (step + 1) % 1000 == 0:
        avg = float(np.mean(loss_log[-1000:]))
        print(f"step {step+1}/{FINAL_STEPS_NEG} avg_loss(last1000)={avg:.4f}")

t1 = time.time()
train_time_neg = t1 - t0
final_avg_loss_neg = float(np.mean(loss_log[-1000:]))

print("\nSkip-gram NEG DONE")
print("time_sec:", train_time_neg)
print("final_avg_loss:", final_avg_loss_neg)

neg_embeddings = model_neg.in_embed.weight.detach().to("cpu").numpy()
np.save("embeddings/sg_neg_embeddings.npy", neg_embeddings)
print("saved: sg_neg_embeddings.npy", neg_embeddings.shape)

In [ ]:
from collections import defaultdict

GLOVE_WINDOW = 2

t0 = time.time()
cooc = defaultdict(float)

for si, sent in enumerate(corpus_ids):
    n = len(sent)
    for i, wi in enumerate(sent):
        start = max(0, i - GLOVE_WINDOW)
        end = min(n, i + GLOVE_WINDOW + 1)
        for j in range(start, end):
            if j == i:
                continue
            wj = sent[j]
            dist = abs(j - i)
            cooc[(wi, wj)] += 1.0 / dist

    if (si + 1) % 1000 == 0:
        print(f"processed sentences: {si+1}/{len(corpus_ids)}; cooc_pairs={len(cooc)}")

t1 = time.time()
print("\nCo-occurrence build DONE")
print("num_pairs:", len(cooc))
print("time_sec:", t1 - t0)

In [ ]:
import math

# Convert cooc dict to arrays
pairs = np.array(list(cooc.keys()), dtype=np.int64)
counts = np.array(list(cooc.values()), dtype=np.float32)
N = len(counts)
print("Training pairs N:", N)

# GloVe hyperparams
EMB_DIM = 50
X_MAX = 100.0
ALPHA = 0.75
BATCH_SIZE_G = 2048
EPOCHS_G = 5
LR_G = 0.05

class GloVe(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.w = nn.Embedding(vocab_size, emb_dim)
        self.w_tilde = nn.Embedding(vocab_size, emb_dim)
        self.b = nn.Embedding(vocab_size, 1)
        self.b_tilde = nn.Embedding(vocab_size, 1)

    def forward(self, i_ids, j_ids):
        wi = self.w(i_ids)
        wj = self.w_tilde(j_ids)
        bi = self.b(i_ids).squeeze(1)
        bj = self.b_tilde(j_ids).squeeze(1)
        return (wi * wj).sum(dim=1) + bi + bj

glove = GloVe(vocab_size=vocab_size, emb_dim=EMB_DIM).to(device)
opt = optim.Adagrad(glove.parameters(), lr=LR_G)

# Precompute weights
fx = np.minimum((counts / X_MAX) ** ALPHA, 1.0).astype(np.float32)
logx = np.log(counts + 1e-8).astype(np.float32)

pairs_t = torch.tensor(pairs, dtype=torch.long, device=device)
fx_t = torch.tensor(fx, dtype=torch.float32, device=device)
logx_t = torch.tensor(logx, dtype=torch.float32, device=device)

t0 = time.time()
glove.train()

for epoch in range(EPOCHS_G):
    perm = torch.randperm(N, device=device)
    total_loss = 0.0
    num_batches = 0

    for start in range(0, N, BATCH_SIZE_G):
        idx = perm[start:start+BATCH_SIZE_G]
        ij = pairs_t[idx]
        i_ids = ij[:, 0]
        j_ids = ij[:, 1]

        pred = glove(i_ids, j_ids)
        diff = pred - logx_t[idx]
        loss = (fx_t[idx] * diff * diff).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += float(loss.item())
        num_batches += 1

    avg_loss = total_loss / max(1, num_batches)
    print(f"epoch {epoch+1}/{EPOCHS_G} avg_loss={avg_loss:.4f}")

t1 = time.time()
train_time_glove = t1 - t0
print("\nGloVe DONE")
print("time_sec:", train_time_glove)

# Final embedding = w + w_tilde
glove_embeddings = (glove.w.weight.detach() + glove.w_tilde.weight.detach()).to("cpu").numpy()
np.save("embeddings/glove_embeddings.npy", glove_embeddings)
print("saved: glove_embeddings.npy", glove_embeddings.shape)

In [ ]:
import json

# Save vocabulary
with open("data/word2id.json", "w") as f:
    json.dump(word2id, f)

# Save training results
results = {
    'skipgram': {
        'window_size': WINDOW_SIZE,
        'training_time': train_time,
        'final_loss': final_avg_loss
    },
    'skipgram_neg': {
        'window_size': WINDOW_SIZE,
        'training_time': train_time_neg,
        'final_loss': final_avg_loss_neg
    },
    'glove': {
        'window_size': GLOVE_WINDOW,
        'training_time': train_time_glove,
        'final_loss': avg_loss
    }
}

with open("results/training_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved!")
print("\nSummary:")
for model, res in results.items():
    print(f"{model}: time={res['training_time']:.2f}s, loss={res['final_loss']:.4f}")